In [2]:
import numpy as np
import matplotlib.pyplot as plt

matrix = np.array([
    [6, 18, 5],
    [17, 13, 15],
    [9, 13, 19]
])

def analytical_method(C):
    C_inv = np.linalg.inv(C)
    total_sum = np.sum(C_inv)
    v = 1 / total_sum

    row_sums = np.sum(C_inv, axis=1)
    x = row_sums / total_sum
    col_sums = np.sum(C_inv, axis=0)
    y = col_sums / total_sum

    return v, x, y

def solve_brown_robinson(C, target_eps, max_iter=10000):
    m, n = C.shape

    acc_win_A = [0.0, 0.0, 0.0]
    acc_loss_B = [0.0, 0.0, 0.0]
    counts_A = [0, 0, 0]
    counts_B = [0, 0, 0]

    v_up_history = []
    v_low_history = []
    log = []

    k = 1
    curr_A = 0
    curr_B = 0
    epsilon = 1.0

    # Таблица
    header = f"{'№':<6} {'A':<6} {'B':<6} {'Выигрыш A (x1,x2,x3)':<25} {'Проигрыш B (y1,y2,y3)':<25} {'v_up':<8} {'v_low':<8}"
    log.append(header)
    log.append("-" * 90)

    while epsilon > target_eps and k <= max_iter:
        counts_A[curr_A] += 1
        counts_B[curr_B] += 1

        acc_win_A[0] += C[0][curr_B]
        acc_win_A[1] += C[1][curr_B]
        acc_win_A[2] += C[2][curr_B]

        acc_loss_B[0] += C[curr_A][0]
        acc_loss_B[1] += C[curr_A][1]
        acc_loss_B[2] += C[curr_A][2]

        v_up_k = max(acc_win_A) / k
        v_low_k = min(acc_loss_B) / k

        v_up_history.append(v_up_k)
        v_low_history.append(v_low_k)

        epsilon = min(v_up_history) - max(v_low_history)

        win_str = f"{int(acc_win_A[0])}, {int(acc_win_A[1])}, {int(acc_win_A[2])}"
        loss_str = f"{int(acc_loss_B[0])}, {int(acc_loss_B[1])}, {int(acc_loss_B[2])}"

        row = f"{k:<6} x{curr_A+1:<5} y{curr_B+1:<5} {win_str:<25} {loss_str:<25} {v_up_k:<8.4f} {v_low_k:<8.4f}"
        log.append(row)

        curr_A = acc_win_A.index(max(acc_win_A))
        curr_B = acc_loss_B.index(min(acc_loss_B))

        k += 1

    steps = k - 1
    x_strat = [round(c / steps, 4) for c in counts_A]
    y_strat = [round(c / steps, 4) for c in counts_B]

    v_res = (min(v_up_history) + max(v_low_history)) / 2

    full_log = "\n".join(log)
    with open("brown_robinson_log.txt", "w", encoding="utf-8") as f:
        f.write(full_log)

    return v_res, x_strat, y_strat, v_up_history, v_low_history, steps

def print_comparison(analytical_v, analytical_x, analytical_y,
                     brown_v, brown_x, brown_y, iterations):

    print("\n" + "-" * 70)
    print("ПОГРЕШНОСТИ:")
    print(f"  Цена игры:      |{analytical_v - brown_v:.4f}|")
    print(f"  Количество итераций: {iterations}")
def main():

    print("АНАЛИТИЧЕСКИЙ МЕТОД")
    v_analytical, x_analytical, y_analytical = analytical_method(matrix)
    print(f"Цена игры: {v_analytical:.4f}")
    print(f"Стратегия A: {[round(x,4) for x in x_analytical]}")
    print(f"Стратегия B: {[round(y,4) for y in y_analytical]}")

    print("\nЧИСЛЕННЫЙ МЕТОД БРАУНА-РОБИНСОНА")
    target_eps = 0.1
    print(f"Целевая погрешность: ε = {target_eps}")
    print("(первые итерации см. в файле brown_robinson_log.txt)")

    v_brown, x_brown, y_brown, v_up_hist, v_low_hist, iterations = solve_brown_robinson(
        matrix, target_eps
    )

    print(f"\nРЕЗУЛЬТАТ после {iterations} итераций:")
    print(f"Цена игры: {v_brown:.4f}")
    print(f"Стратегия A: {x_brown}")
    print(f"Стратегия B: {y_brown}")

    # Сравнение
    print_comparison(v_analytical, x_analytical, y_analytical,
                     v_brown, x_brown, y_brown, iterations)

if __name__ == "__main__":
    main()


АНАЛИТИЧЕСКИЙ МЕТОД
Цена игры: 13.8696
Стратегия A: [np.float64(0.1087), np.float64(0.6739), np.float64(0.2174)]
Стратегия B: [np.float64(0.1739), np.float64(0.6739), np.float64(0.1522)]

ЧИСЛЕННЫЙ МЕТОД БРАУНА-РОБИНСОНА
Целевая погрешность: ε = 0.1
(первые итерации см. в файле brown_robinson_log.txt)

РЕЗУЛЬТАТ после 124 итераций:
Цена игры: 13.8692
Стратегия A: [0.1774, 0.6694, 0.1532]
Стратегия B: [0.1129, 0.5806, 0.3065]

----------------------------------------------------------------------
ПОГРЕШНОСТИ:
  Цена игры:      |0.0003|
  Количество итераций: 124
